In [ ]:
from vnlegal_rag_v2.data import DataPreparationPipeline

pipeline = DataPreparationPipeline(
    raw_path="data/raw",
    processed_path="data/processed",
    segmentation_method="underthesea",
    
    overwrite=True,
)

pipeline.run()

## Models

In [ ]:
from vnlegal_rag_v2.utils.device import get_device
from vnlegal_rag_v2.models import BiEncoderModel, CrossEncoderModel

# Device detection
print(f"Device: {get_device()}")

# Bi-Encoder test
bi_encoder = BiEncoderModel("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
texts = ["Điều luật số 1", "Nghị định về an ninh quốc gia"]
embeddings = bi_encoder.encode(texts)
print(f"Bi-Encoder embeddings shape: {embeddings.shape}")

# Cross-Encoder test
cross_encoder = CrossEncoderModel("cross-encoder/ms-marco-MiniLM-L-6-v2")
pairs = [["điều luật", "Điều luật số 1"], ["điều luật", "Nghị định về an ninh quốc gia"]]
scores = cross_encoder.predict(pairs)
print(f"Cross-Encoder scores: {scores}")


## Evaluation

In [ ]:
from vnlegal_rag_v2.evaluation import (
    Evaluator,
    mrr_at_k,
    success_at_k,
    recall_at_k,
    precision_at_k,
    f1_at_k,
)

# Simulated predictions: 3 queries
predictions = [
    [1, 2, 3, 4, 5],      # query 1: hit at rank 1
    [5, 4, 1, 2, 3],      # query 2: first hit at rank 3
    [9, 8, 7, 6, 10],     # query 3: no hit
]
relevant = [
    [1, 10],
    [1, 2],
    [1, 2],
]

# Default: MRR@10 + Success@10
evaluator = Evaluator(predictions, relevant)
print('Default:', evaluator.evaluate())

# Custom metrics
results = evaluator.evaluate([
    ('mrr@5', mrr_at_k, 5),
    ('success@5', success_at_k, 5),
    ('recall@5', recall_at_k, 5),
    ('precision@5', precision_at_k, 5),
    ('f1@5', f1_at_k, 5),
    ('success@1', success_at_k, 1),
])
for k, v in results.items():
    print(f'  {k} = {v:.4f}')

## Dense Retrieval

In [ ]:
from vnlegal_rag_v2.models import BiEncoderModel
from vnlegal_rag_v2.retrieval import DenseRetriever

model = BiEncoderModel('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', device='cpu')
retriever = DenseRetriever(model)

documents = [
    'Điều 1: Luật hình sự',
    'Điều 2: Nghị định về an ninh',
    'Điều 3: Luật dân sự',
    'Điều 4: Luật đất đai',
    'Điều 5: Nghị định thuế',
]
cids = [101, 102, 103, 104, 105]

retriever.index(documents, cids)

queries = ['luật hình sự', 'nghị định']
results = retriever.retrieve(queries, top_k=3)

for q, r in zip(queries, results):
    print(f'{q} -> {r}')

In [ ]:
# Dense Retrieval with segmentation (Vietnamese models)
from vnlegal_rag_v2.models import BiEncoderModel
from vnlegal_rag_v2.retrieval import DenseRetriever

model = BiEncoderModel("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device="cpu")
retriever = DenseRetriever(model, segmentation_method="pyvi")

documents = [
    "Điều 1: Luật hình sự quy định về tội phạm",
    "Điều 2: Nghị định về an ninh quốc gia",
    "Điều 3: Luật dân sự và hợp đồng",
    "Điều 4: Luật đất đai và quy hoạch",
    "Điều 5: Nghị định thuế thu nhập cá nhân",
]
cids = [101, 102, 103, 104, 105]

retriever.index(documents, cids)

queries = ["luật hình sự", "nghị định"]
results = retriever.retrieve(queries, top_k=3)

for q, r in zip(queries, results):
    print(f"{q} -> {r}")

# Also test no segmentation (default)
retriever_no_seg = DenseRetriever(model)
retriever_no_seg.index(documents, cids)
results2 = retriever_no_seg.retrieve(queries, top_k=3)
print("No segmentation:")
for q, r in zip(queries, results2):
    print(f"{q} -> {r}")


## Sparse Retrieval

In [ ]:
from vnlegal_rag_v2.retrieval import BM25Retriever, TFIDFRetriever

documents = [
    'Điều 1: Luật hình sự quy định về tội phạm',
    'Điều 2: Nghị định về an ninh quốc gia',
    'Điều 3: Luật dân sự và hợp đồng',
    'Điều 4: Luật đất đai và quy hoạch',
    'Điều 5: Nghị định thuế thu nhập cá nhân',
]
cids = [101, 102, 103, 104, 105]

# BM25
bm25 = BM25Retriever()
bm25.index(documents, cids)
results = bm25.retrieve(['luật hình sự', 'nghị định thuế'], top_k=3)
print('BM25:')
for q, r in zip(['luật hình sự', 'nghị định thuế'], results):
    print(f'  {q} -> {r}')

# TF-IDF
tfidf = TFIDFRetriever()
tfidf.index(documents, cids)
results = tfidf.retrieve(['luật hình sự', 'nghị định thuế'], top_k=3)
print('TF-IDF:')
for q, r in zip(['luật hình sự', 'nghị định thuế'], results):
    print(f'  {q} -> {r}')

## Hybrid Retrieval (RRF)

In [ ]:
from vnlegal_rag_v2.retrieval import BM25Retriever, TFIDFRetriever, HybridRetriever

documents = [
    'Điều 1: Luật hình sự quy định về tội phạm',
    'Điều 2: Nghị định về an ninh quốc gia',
    'Điều 3: Luật dân sự và hợp đồng',
    'Điều 4: Luật đất đai và quy hoạch',
    'Điều 5: Nghị định thuế thu nhập cá nhân',
]
cids = [101, 102, 103, 104, 105]

hybrid = HybridRetriever(
    retrievers=[BM25Retriever(), TFIDFRetriever()],
    weights=[0.5, 0.5],
)
hybrid.index(documents, cids)
results = hybrid.retrieve(['luật hình sự', 'nghị định thuế'], top_k=3)

print('Hybrid (BM25 + TF-IDF, RRF):')
for q, r in zip(['luật hình sự', 'nghị định thuế'], results):
    print(f'  {q} -> {r}')

## Reranking

In [ ]:
from vnlegal_rag_v2.models import CrossEncoderModel
from vnlegal_rag_v2.reranking import CrossEncoderReranker

ce_model = CrossEncoderModel('cross-encoder/ms-marco-MiniLM-L-6-v2', device='cpu')
reranker = CrossEncoderReranker(ce_model)

documents = [
    'Điều 1: Luật hình sự quy định về tội phạm',
    'Điều 2: Nghị định về an ninh quốc gia',
    'Điều 3: Luật dân sự và hợp đồng',
    'Điều 4: Luật đất đai và quy hoạch',
    'Điều 5: Nghị định thuế thu nhập cá nhân',
]
cids = [101, 102, 103, 104, 105]

candidate_cids = [
    [105, 103, 101],
    [105, 102, 104],
]

results = reranker.rerank(
    queries=['luật hình sự', 'nghị định thuế'],
    candidate_cids=candidate_cids,
    documents=documents,
    cids=cids,
    top_k=3,
    show_progress_bar=False,
)

for q, r in zip(['luật hình sự', 'nghị định thuế'], results):
    print(f'{q} -> {r}')

## Pipeline

In [ ]:
from vnlegal_rag_v2.data import load_processed, extract_corpus, extract_queries

# Sample small subset for fast testing
eval_df = load_processed('data/processed', 'eval.csv').sample(50, random_state=42)
documents, cids = extract_corpus(eval_df)
questions, relevant = extract_queries(eval_df)
print(f'Docs: {len(documents)}, Queries: {len(questions)}')

In [ ]:
from vnlegal_rag_v2.models import BiEncoderModel
from vnlegal_rag_v2.retrieval import DenseRetriever
from vnlegal_rag_v2.pipeline import RAGPipeline

bi = BiEncoderModel('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', device='cpu')
dense_pipeline = RAGPipeline(
    retriever=DenseRetriever(bi),
    top_k_retrieval=20,
)
dense_pipeline.index(documents, cids)

scores = dense_pipeline.evaluate(questions, relevant)
print('Dense-only:')
for k, v in scores.items():
    print(f'  {k} = {v:.4f}')

In [ ]:
from vnlegal_rag_v2.models import BiEncoderModel, CrossEncoderModel
from vnlegal_rag_v2.retrieval import BM25Retriever, DenseRetriever, HybridRetriever
from vnlegal_rag_v2.reranking import CrossEncoderReranker
from vnlegal_rag_v2.pipeline import RAGPipeline

bi = BiEncoderModel('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', device='cpu')
ce = CrossEncoderModel('cross-encoder/ms-marco-MiniLM-L-6-v2', device='cpu')

full_pipeline = RAGPipeline(
    retriever=HybridRetriever(
        retrievers=[BM25Retriever(), DenseRetriever(bi)],
        weights=[0.4, 0.6],
    ),
    rerankers=[CrossEncoderReranker(ce)],
    top_k_retrieval=20,
    top_k_rerank=5,
)
full_pipeline.index(documents, cids)

scores = full_pipeline.evaluate(questions, relevant)
print('Hybrid + Cross-Encoder:')
for k, v in scores.items():
    print(f'  {k} = {v:.4f}')

In [ ]:
# Compare: retrieval-only vs retrieve+rerank on same data
from vnlegal_rag_v2.evaluation import Evaluator

# Reuse full_pipeline from above (already indexed)
candidates = full_pipeline.retrieve(questions, top_k=20)
reranked = full_pipeline.rerank(questions, candidates, top_k=5)

ret_scores = Evaluator(candidates, relevant).evaluate()
final_scores = Evaluator(reranked, relevant).evaluate()

print('Retrieval@20: ', ret_scores)
print('After rerank@5: ', final_scores)

In [ ]:
# Sanity check with toy data (no models needed)
from vnlegal_rag_v2.retrieval import BM25Retriever
from vnlegal_rag_v2.pipeline import RAGPipeline

toy_docs = [
    'Điều 1: Luật hình sự quy định về tội phạm',
    'Điều 2: Nghị định về an ninh quốc gia',
    'Điều 3: Luật dân sự và hợp đồng',
    'Điều 4: Luật đất đai và quy hoạch',
    'Điều 5: Nghị định thuế thu nhập cá nhân',
]
toy_cids = [101, 102, 103, 104, 105]
toy_queries = ['luật hình sự', 'nghị định thuế']
toy_relevant = [[101], [105]]

toy_pipeline = RAGPipeline(
    retriever=BM25Retriever(),
    top_k_retrieval=3,
)
toy_pipeline.index(toy_docs, toy_cids)

preds = toy_pipeline.query(toy_queries)
print('Predictions:')
for q, p in zip(toy_queries, preds):
    print(f'  {q} -> {p}')

scores = toy_pipeline.evaluate(toy_queries, toy_relevant)
print(f'Scores: {scores}')

## Training

In [ ]:
from vnlegal_rag_v2.training import BiEncoderTrainer

# Prepare small subset for quick test
import os, shutil
from vnlegal_rag_v2.data import load_processed

os.makedirs("data/test_training", exist_ok=True)
train_small = load_processed("data/processed", "train.csv").head(200)
eval_small = load_processed("data/processed", "eval.csv").head(50)
train_small.to_csv("data/test_training/train.csv", index=False)
eval_small.to_csv("data/test_training/eval.csv", index=False)

bi_trainer = BiEncoderTrainer(
    model_name_or_path="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device="cpu",
)

bi_trainer.train(
    data_path="data/test_training",
    num_epochs=1,
    batch_size=16,
    fp16=False,
)

# Verify: load best model
from vnlegal_rag_v2.models import BiEncoderModel
trained_bi = BiEncoderModel("models/bi-encoder/best", device="cpu")
embs = trained_bi.encode(["luật hình sự", "nghị định thuế"])
print(f"Trained Bi-Encoder embeddings shape: {embs.shape}")

# Show structure
for d in os.listdir("models/bi-encoder"):
    print(f"  models/bi-encoder/{d}/")

shutil.rmtree("data/test_training", ignore_errors=True)


In [ ]:
from vnlegal_rag_v2.training import CrossEncoderTrainer

# Prepare small subset for quick test
import os, shutil
from vnlegal_rag_v2.data import load_processed

os.makedirs("data/test_training", exist_ok=True)
train_small = load_processed("data/processed", "train.csv").head(200)
eval_small = load_processed("data/processed", "eval.csv").head(50)
train_small.to_csv("data/test_training/train.csv", index=False)
eval_small.to_csv("data/test_training/eval.csv", index=False)

ce_trainer = CrossEncoderTrainer(
    model_name_or_path="cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cpu",
)

ce_trainer.train(
    data_path="data/test_training",
    num_epochs=1,
    batch_size=16,
    num_negatives_per_positive=1,
)

# Verify: load best model
from vnlegal_rag_v2.models import CrossEncoderModel
trained_ce = CrossEncoderModel("models/cross-encoder/best", device="cpu")
scores = trained_ce.predict([["luật hình sự", "Điều 1: Luật hình sự"], ["luật hình sự", "Điều 5: Nghị định thuế"]])
print(f"Trained Cross-Encoder scores: {scores}")

# Show structure
for d in os.listdir("models/cross-encoder"):
    print(f"  models/cross-encoder/{d}/")

shutil.rmtree("data/test_training", ignore_errors=True)


## Negative Mining

In [ ]:
from vnlegal_rag_v2.mining import NegativeMiner
from vnlegal_rag_v2.retrieval.sparse import BM25Retriever
from vnlegal_rag_v2.data.loaders import load_processed, extract_corpus, extract_queries

# Small subset for quick test
train_df = load_processed('data/processed', 'train.csv')
documents, cids = extract_corpus(train_df)
queries, relevant = extract_queries(train_df)

n_docs = 1000
sub_docs, sub_cids = documents[:n_docs], cids[:n_docs]
sub_q, sub_r = queries[:20], relevant[:20]

retriever = BM25Retriever()
miner = NegativeMiner(retriever, sub_docs, sub_cids)

for strategy in ['easy', 'moderate', 'semi_hard', 'hard']:
    result = miner.mine(
        sub_q, sub_r,
        strategy=strategy, num_negatives=3, top_k=20,
        margin=5 if strategy == 'semi_hard' else None,
    )
    queries_hit = len({r['query'] for r in result})
    print(f'{strategy:12s} → {len(result):4d} negatives ({queries_hit} queries)')

# Sample output
print()
for r in miner.mine(sub_q[:2], sub_r[:2], strategy='hard', num_negatives=2, top_k=20):
    print(f'  q="{r["query"][:50]}" neg_cid={r["negative_cid"]}')


In [ ]:
# Static methods — work without a retriever instance
from vnlegal_rag_v2.mining import NegativeMiner

corpus_cids = list(range(1, 21))
relevant = [[1, 2], [3, 4], [5, 6]]

# Easy: random from corpus
easy = NegativeMiner.random_negatives(corpus_cids, relevant, 3, seed=42)
print(f'easy:     {easy}')

# Hard: top-ranked false positives from predictions
preds = [[1, 5, 3, 8, 9, 10], [3, 1, 7, 2, 11, 12], [5, 2, 1, 13, 14, 15]]
hard = NegativeMiner.from_predictions(preds, relevant, 3, strategy='hard')
print(f'hard:     {hard}')

# Moderate: random from predictions
moderate = NegativeMiner.from_predictions(preds, relevant, 3, strategy='moderate', seed=42)
print(f'moderate: {moderate}')

# Semi-hard: just after last relevant in ranked list
semi = NegativeMiner.semi_hard_negatives(preds, relevant, 3, margin=5)
print(f'semi_hard: {semi}')


In [ ]:
# Save hard negatives to CSV for Cross-Encoder training
import os

os.makedirs('data/negatives', exist_ok=True)
df = miner.mine_to_csv(
    sub_q, sub_r,
    'data/negatives/hard_negatives.csv',
    strategy='hard', num_negatives=3, top_k=20,
)
print(f'Saved {len(df)} rows')
print(df.head(6).to_string(max_colwidth=50))


## Factory & Configs

In [ ]:
from vnlegal_rag_v2.factory import build_pipeline
import yaml

# BM25 pipeline from config
config = yaml.safe_load(open("configs/pipeline/bm25_only.yaml"))
pipeline = build_pipeline(config)
print(f"retriever: {type(pipeline.retriever).__name__}")
print(f"rerankers: {pipeline.rerankers}")
print(f"top_k_retrieval: {pipeline.top_k_retrieval}")


In [ ]:
# End-to-end BM25 on tiny data via run_pipeline.py logic
from vnlegal_rag_v2.data.loaders import load_processed, extract_corpus, extract_queries
from vnlegal_rag_v2.factory import build_pipeline
import yaml

config = yaml.safe_load(open("configs/pipeline/bm25_only.yaml"))
eval_df = load_processed("data/processed", "eval.csv").head(10)
queries, relevant = extract_queries(eval_df)
documents, cids = extract_corpus(eval_df)

pipeline = build_pipeline(config)
pipeline.index(documents, cids)
scores = pipeline.evaluate(queries, relevant)

print("BM25 pipeline on 10 docs:")
for k, v in scores.items():
    print(f"  {k} = {v:.4f}")


In [ ]:
# Verify all configs parse correctly
import yaml, glob

for f in sorted(glob.glob("configs/**/*.yaml", recursive=True)):
    config = yaml.safe_load(open(f))
    name = config.get("name", f)
    method = config.get("retrieval", {}).get("method", "data/train")
    print(f"  {f}: name={name}, method={method}")
